**23 | Unified Matched Benchmark: Seven HARTH Activity Classifiers**

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Research aim:** Compare Random Forest, Standard CNN, Weighted CNN, TCN, Multi-Scale CNN, 1D-ResNet and Time-Series Transformer in one source-row- and participant-matched eight-class development experiment. All classifiers use precisely the same subject-held-out GroupKFold Validation windows. This is an **exploratory development benchmark**, not independent Test generalisation or evidence of a clinically validated activity recognition system.

**Resumability:** Each model/fold job saves original source-row IDs, true labels, predictions, hyperparameters, architecture-source fingerprint and timing in an ignored local JSON file. Interrupted training can resume by rerunning the training cell. Results are not displayed as complete until every expected job passes source-alignment validation. **Only actual executed notebook outputs**, not local cache or untrained model results, are pushed to GitHub.

**01. Scientific Question, Model Families and Evaluation Boundary**

We ask whether the four temporal architectures from Stage 22 outperform the existing RF and simple CNN baselines on the *same* bounded HARTH development data. The predefined target codes are `1–8`; cycling and undocumented labels are removed **after** time-safe window formation. The dataset contains 29 Development participants. Original Test subjects S023/S032 were inspected early in the project but are excluded from all subsequent training, selection and scoring.

**Important comparability distinctions:** RF receives 30 deterministic statistical features extracted from the identical raw source windows, whereas neural models receive all 6×100 source samples. Neural networks share 400 Optimiser Steps, Adam, batch size, learning rate, source windows and folds; Random Forest uses its established 120-tree configuration. Fixed Steps do *not* equalise FLOPs, parameter counts, data exposure, training convergence or runtime. Models are evaluated using fixed-eight-class Macro F1 rather than overall accuracy alone.

In [ ]:
# ==================== 01. Imports, Determinism and Scientific Contracts ====================

# 1. Import general tools and locate existing research code without packaging it.
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
root=Path.cwd().resolve()
while not (root/"code/python/src/core/data_loading.py").is_file():
    if root==root.parent: raise RuntimeError("Open notebook inside the repository.")
    root=root.parent
sys.path.insert(0,str(root/"code/python/src"))
sys.path.insert(0,str(root/"studies/wearable_activity_recognition/src"))

# 2. Import the pinned HARTH source adapter and shared model/evaluation modules.
from harth_data import (
    HARTH_SUBJECTS, HARTH_REVISION, SENSOR_COLUMNS, ACTIVITY_NAMES,
    prepare_harth_window_cohort
)
from core.temporal_benchmark import (
    MODEL_NAMES, PROTOCOL_VERSION, make_benchmark_plan,
    run_benchmark_job, summarise_benchmark_results
)
from core.feature_engineering import extract_signal_features

# 3. Freeze the exact comparison scope and reserve original Test participants.
RESERVED_TEST={"S023","S032"}
DEVELOPMENT_SUBJECTS=sorted(set(HARTH_SUBJECTS)-RESERVED_TEST)
CLASSES=tuple(range(1,9))
FOLDS=3
MAX_WINDOWS_PER_SUBJECT=500
MAX_STEPS=400
BATCH_SIZE=128
SEED=42
# 4. Require an active Colab GPU before scheduling 18 expensive neural fits.
#    Do not silently fall back to CPU during this benchmark.
REQUIRE_CUDA = True
if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is unavailable. In Google Colab select Runtime > "
        "Change runtime type > Hardware accelerator > GPU, then reconnect."
    )
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 5. Keep full float32 precision without TF32 approximation where supported.
#    Deterministic cuDNN controls also remain active inside each training job.
if DEVICE == "cuda":
    torch.set_float32_matmul_precision("highest")
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    gpu_properties = torch.cuda.get_device_properties(0)
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEMORY_GB = round(gpu_properties.total_memory / 1024**3, 2)
else:
    GPU_NAME = "No CUDA GPU"
    GPU_MEMORY_GB = 0.0

# 6. Resume completed fold/model jobs from a source-verified local cache.
#    Colab /content storage is temporary if the runtime is reset.
CACHE_DIR=root/"data/processed/harth_stage23_matched_benchmark"
assert len(DEVELOPMENT_SUBJECTS)==29 and not(set(DEVELOPMENT_SUBJECTS)&RESERVED_TEST)
assert len(MODEL_NAMES)==7

print("\n"+"="*70)
print("STAGE 23 | SEVEN-MODEL MATCHED DEVELOPMENT BENCHMARK")
print("="*70)
print("Pinned official HARTH revision:",HARTH_REVISION)
print("Device:",DEVICE,"| PyTorch:",torch.__version__)
print("GPU:",GPU_NAME,"| Visible GPU memory (GiB):",GPU_MEMORY_GB)
print("CUDA runtime version:",torch.version.cuda)
print("CUDA active for all six neural models:",DEVICE=="cuda")
print("Original Test excluded:",sorted(RESERVED_TEST))
print("Class codes:",CLASSES)
print("Models:",", ".join(MODEL_NAMES))
print("Neural training update budget:",MAX_STEPS)
print("="*70)

**02. Recreate Original Real-Source Development Windows**

We reconstruct the *same* 500-window-per-participant bounded cohort used in previous Stage 13 and 17 comparisons, rather than making an incomparable new sample. Each individual public CSV is processed separately; windows have 100 samples at nominal 50 Hz, 50-sample stride, zero annotation impurity and timestamp continuity. Sample positions are selected evenly over eligible chronological windows without using activity labels. The expected 29-person cohort contains 14,201 raw windows. Because participants are independent groups and windows overlap, model scoring is subject-held-out but individual windows are correlated.

In [ ]:
# ==================== 02. Verified Real-Source Signal Cohort ====================

# 1. Recreate the bounded development cohort using the exact established adapter.
X,y,metadata,overview=prepare_harth_window_cohort(
    subjects=DEVELOPMENT_SUBJECTS,
    data_dir=root/"data/raw/harth_v2",
    task_classes=CLASSES,max_windows_per_subject=MAX_WINDOWS_PER_SUBJECT,
    window_size=100,step_size=50,sampling_rate=50,
)
assert X.shape==(14201,6,100)
assert len(y)==len(metadata) and metadata["subject_id"].nunique()==29
assert not(set(metadata["subject_id"])&RESERVED_TEST)

# 2. Display actual class frequencies and independently contributing participants.
counts=pd.crosstab(
    metadata["subject_id"],metadata["label"]
).reindex(index=DEVELOPMENT_SUBJECTS,columns=CLASSES,fill_value=0)
coverage=pd.DataFrame({
    "activity":[ACTIVITY_NAMES[c] for c in CLASSES],
    "windows":counts.sum(axis=0).to_numpy(),
    "supporting_subjects":(counts>0).sum(axis=0).to_numpy(),
})
display(overview[["subject_id","eligible_windows","sampled_windows"]])
display(coverage)
print("True raw IMU tensor:",X.shape)
print("Window class codes:",sorted(np.unique(y).tolist()))

**03. Freeze Three Participant-Disjoint Folds and Source Fingerprint**

Every model receives an identical set of validation source row IDs for each fold. `GroupKFold` partitions by **participant**, not overlapping window. All development rows are validated exactly once by each classifier, and the eight-class denominator stays fixed even when a given person lacks one or more activities. A SHA256 fingerprint of all actual source windows, labels, identities and fixed protocol guards against accidentally restoring predictions from a different dataset.

In [ ]:
# ==================== 03. Matched GroupKFold and Source Provenance ====================

# 1. Build one source-fingerprinted evaluation plan for every architecture.
plan=make_benchmark_plan(
    windows=X,labels=y,subject_ids=metadata["subject_id"].to_numpy(),
    classes=CLASSES,n_splits=FOLDS,forbidden_subjects=RESERVED_TEST
)

# 2. Display group identity, sample counts and class availability for each fold.
fold_design=[]
for fold,tr,va in plan["folds"]:
    tr_groups=set(plan["groups"][tr])
    va_groups=set(plan["groups"][va])
    assert not(tr_groups&va_groups)
    fold_design.append({
        "fold":fold,"train_subjects":len(tr_groups),
        "validation_subjects":len(va_groups),
        "train_windows":len(tr),"validation_windows":len(va),
        "train_classes":len(np.unique(y[tr])),
        "validation_classes":len(np.unique(y[va])),
        "validation_subject_ids":", ".join(sorted(va_groups)),
    })
display(pd.DataFrame(fold_design))
print("Source SHA256:",plan["fingerprint"])
print("Complete model-by-fold jobs required:",len(MODEL_NAMES)*FOLDS)

**04. Fix Model Hyperparameters and Training Budget**

All six neural networks start from newly initialised weights in each fold and have precisely **400 Adam optimiser updates** with batch size 128 and learning rate `0.001`. The Standard CNN uses ordinary Cross-Entropy; Weighted CNN and all four advanced neural models use **inverse-square-root class loss weighting computed exclusively from Fold Train labels**. Each model's channel mean and SD are also computed from Fold Train windows. The RF retains 120 trees, minimum leaf size three and balanced bootstrap class weighting with source-window statistical features.

**Interpretation:** Comparing all advanced architectures under one weighted-loss scheme controls the loss setting for those models, but differences from Standard CNN also reflect loss weighting. The Weighted CNN is the primary neural architecture reference. Equal updates are a practical control, not an assertion of equal computation.

In [ ]:
# ==================== 04. Seven-Model Protocol Summary ====================

# 1. State what is shared and what differs by model family.
protocol=pd.DataFrame([
    {"model":name,"representation":
      "30 deterministic signal features" if name=="Random Forest" else "6 x 100 raw IMU",
     "training":
      "120-tree balanced-subset RF" if name=="Random Forest"
      else "400 Adam updates; inverse-sqrt weighted CE"
      if name!="Standard CNN" else "400 Adam updates; unweighted CE"}
    for name in MODEL_NAMES
])
display(protocol)

# 2. Extract the same deterministic statistics for RF, without fitting a scaler.
features=extract_signal_features(X,channel_names=SENSOR_COLUMNS)
assert features.shape==(len(X),30)
assert np.isfinite(features.to_numpy()).all()
rf_features=features.to_numpy()
print("RF feature matrix:",rf_features.shape)
print("All CNN/Transformer input rows matched with RF source rows: YES")

**05. Run All 21 Matched Model-by-Fold Jobs with Safe Resumption**

There are **7 classifiers × 3 folds = 21 jobs**: six neural architectures × three held-out folds (18 neural fits), plus three RF fits. Neural models use 400 updates each; on a CPU this can take a considerable time, especially for the Transformer. If possible, use a CUDA-enabled PyTorch installation (Stage 22 confirmed GPU availability in the previous execution environment). The cell writes compact per-job JSON predictions and metrics only after successful completion into the ignored `data/processed/` cache. If interrupted, rerun the cell: completed jobs will be verified and reused, and remaining jobs will run. No job can be counted as a finished benchmark result until its source rows and class labels have been checked.

In [ ]:
# ==================== 05. Matched Benchmark Training and Safe Resume ====================

# 1. Fix job order so resumed runs process the same source models and folds.
#    Large models will require substantial time; allow this cell to finish.
jobs=[]
for fold,tr,va in plan["folds"]:
    print("\n"+"="*70)
    print(f"FOLD {fold}/{FOLDS}: TRAIN {len(tr):,} | VALIDATION {len(va):,}")
    print("="*70)
    for model_name in MODEL_NAMES:
        # 2. Train only if a fingerprint-validated result is not cached.
        result=run_benchmark_job(
            plan=plan,fold=fold,model_name=model_name,
            cache_dir=CACHE_DIR,feature_matrix=rf_features,
            max_steps=MAX_STEPS,batch_size=BATCH_SIZE,learning_rate=1e-3,
            seed=SEED,device=DEVICE,
        )
        jobs.append(result)

# 3. Require a complete matrix instead of displaying partial results as final.
assert len(jobs)==7*FOLDS
print("\n"+"="*70)
print("ALL 21 SOURCE-MATCHED EXPERIMENT JOBS COMPLETED")
print("Local cache:",CACHE_DIR)
print("Predictions and metrics were generated from actual models: YES")
print("="*70)

**06. Reconstruct the Full Scientific Evidence from Out-of-Fold Predictions**

The reporting function independently recalculates accuracy, balanced accuracy, fixed-eight-class Macro F1, one score per held-out participant, pooled out-of-fold precision/recall/F1 for every class and complete confusion matrices. It refuses incomplete/duplicate jobs and mismatched source-row identities. **Do not** interpret high overall accuracy as evidence of strong minority-class recognition.

In [ ]:
# ==================== 06. Complete Benchmark Results and Fold Reporting ====================

# 1. Recalculate every evaluation metric from aligned held-out predictions.
fold_results,subject_results,class_results,predictions,confusion=(
    summarise_benchmark_results(plan,jobs)
)
display(fold_results[[
    "model","fold","train_subjects","validation_subjects",
    "train_windows","validation_windows","macro_f1",
    "balanced_accuracy","accuracy","train_seconds",
    "validation_inference_seconds","trainable_parameters",
]].round(4))

# 2. Aggregate each model's three held-out folds descriptively.
summary=fold_results.groupby("model",as_index=False).agg(
    macro_f1_mean=("macro_f1","mean"),
    macro_f1_sd=("macro_f1","std"),
    balanced_accuracy_mean=("balanced_accuracy","mean"),
    accuracy_mean=("accuracy","mean"),
    total_train_seconds=("train_seconds","sum"),
    total_inference_seconds=("validation_inference_seconds","sum"),
    parameters=("trainable_parameters","max"),
)
display(summary.sort_values("macro_f1_mean",ascending=False).round(4))
print("All source rows evaluated once per model:",len(X))
print("Independent Validation participant coverage:",len(DEVELOPMENT_SUBJECTS))

**07. Compare Seven Models Across Exactly Matched Held-Out Folds**

Compare each model's fold-level Macro F1 side by side and report its across-fold mean/SD, without treating only three dependent partitions as independent statistical trials. A figure of paired fold scores exposes instability concealed by averages. Random Forest and neural models differ in representation and fitting cost; the comparison remains a practically meaningful end-to-end pipeline benchmark rather than a test of architecture alone.

In [ ]:
# ==================== 07. Paired Fold Metrics and Architecture Comparison ====================

# 1. Plot each fold using the same model order and fixed metric range.
wide=fold_results.pivot(index="model",columns="fold",values="macro_f1"
).reindex(MODEL_NAMES)
display(wide.round(4))
fig,axes=plt.subplots(1,2,figsize=(15,6))
wide.plot(kind="bar",ax=axes[0],rot=35,ylim=(0,1))
axes[0].set(title="Matched three-fold fixed-eight-class Macro F1",
            xlabel="Classifier",ylabel="Macro F1")
for model in MODEL_NAMES:
    rows=fold_results[fold_results["model"]==model].sort_values("fold")
    axes[1].plot(rows["fold"],rows["macro_f1"],marker="o",label=model)
axes[1].set(title="Paired fold variability",xlabel="Held-out fold",
            ylabel="Macro F1",ylim=(0,1),xticks=[1,2,3])
axes[1].legend(fontsize=8)
for ax in axes: ax.grid(axis="y",alpha=0.2)
fig.tight_layout()
plt.show()

**08. Class-Level F1, Shuffling, Stair Activities and Confusion Matrices**

The primary minority-class questions concern Shuffling, Stairs ascending and Stairs descending. A class-level table must show support alongside precision, recall and F1. Pooled out-of-fold metrics are calculated across correlated windows and should be interpreted as descriptive rather than independent person-level outcomes. The confusion matrices show which activities models confuse, retaining the same original class-code order in all seven figures.

In [ ]:
# ==================== 08. Minority-Class Reporting and Confusion Evidence ====================

# 1. Show complete class support and specific difficult-activity diagnostics.
class_results["activity"]=class_results["class_code"].map(ACTIVITY_NAMES)
display(class_results[[
    "model","class_code","activity","support","precision","recall","f1"
]].round(4))
minority=class_results[class_results["class_code"].isin([3,4,5])]
display(minority[["model","activity","support","recall","f1"]].round(4))

# 2. Plot all original task-class F1 values in grouped bars.
class_pivot=class_results.pivot(index="activity",columns="model",values="f1")
class_pivot=class_pivot.reindex([ACTIVITY_NAMES[c] for c in CLASSES])
class_pivot=class_pivot.reindex(columns=MODEL_NAMES)
fig,ax=plt.subplots(figsize=(17,6))
class_pivot.plot(kind="bar",ax=ax,rot=35,ylim=(0,1))
ax.set(title="Seven-model pooled OOF class-level F1",
       xlabel="Documented source activity",ylabel="F1")
ax.legend(fontsize=8,ncol=3)
ax.grid(axis="y",alpha=0.2)
fig.tight_layout()
plt.show()

# 3. Keep a single comparative figure with one confusion panel per model.
fig,axes=plt.subplots(2,4,figsize=(17,9))
for ax,name in zip(axes.flat,MODEL_NAMES):
    matrix=confusion[name]
    ax.imshow(matrix,interpolation="nearest",aspect="auto")
    ax.set(title=name,xlabel="Predicted code",ylabel="True code")
    ax.set_xticks(range(8),CLASSES)
    ax.set_yticks(range(8),CLASSES)
for ax in list(axes.flat)[len(MODEL_NAMES):]:
    ax.axis("off")
fig.suptitle("OOF confusion matrices | identical source class order",y=1.01)
fig.tight_layout()
plt.show()

**09. Participant-Level Fairness of Comparison and Model-Error Variability**

One model score per held-out person reduces the tendency to mistake thousands of overlapping windows for thousands of independent subjects. Our fixed-task Macro F1 includes all eight task codes for each individual, whereas an observed-class F1 only includes activities actually labelled for that participant. Both interpretations matter and must not be conflated. This is an exploratory robustness comparison, not a demographic fairness analysis or a statistical-significance claim.

In [ ]:
# ==================== 09. Participant-Level Results and Paired Scores ====================

# 1. Compare fixed-task participant Macro F1 with explicit independent IDs.
individual=subject_results.pivot(
    index="subject_id",columns="model",values="macro_f1_fixed_task"
).reindex(index=DEVELOPMENT_SUBJECTS,columns=MODEL_NAMES)
display(individual.round(4))
assert individual.notna().all().all()
print("Participants per model:",individual.notna().sum().to_dict())

# 2. Visualise all participants' results consistently in one figure.
fig,ax=plt.subplots(figsize=(16,8))
im=ax.imshow(individual.to_numpy(),aspect="auto",vmin=0,vmax=1)
ax.set_xticks(range(len(MODEL_NAMES)),MODEL_NAMES,rotation=40,ha="right")
ax.set_yticks(range(len(DEVELOPMENT_SUBJECTS)),DEVELOPMENT_SUBJECTS)
ax.set(title="Per-person fixed-eight-class Macro F1",xlabel="Model",
       ylabel="Held-out development participant")
fig.colorbar(im,ax=ax,label="Fixed-eight-class Macro F1")
fig.tight_layout()
plt.show()

**10. Model Complexity, Training Time and Experimental Reproducibility**

Parameter count is meaningful for the six neural architectures, while Random Forest is characterised by its number of trees and training time, not an equivalent neural parameter count. Training time includes batch preparation and optimisation but not public-source download or preprocessing; Validation inference time depends on the local hardware. Caches exist only for successful fold/model runs and are fingerprinted to source arrays and code. They hold predictions, **not reloadable model weights**; a future inference release needs separate final-model training and saved weights in Stage 26.

In [ ]:
# ==================== 10. Engineering Cost and Reproducibility Checks ====================

# 1. Report real machine-relative training and held-out inference time.
costs=fold_results.groupby("model",as_index=False).agg(
    total_train_s=("train_seconds","sum"),
    mean_train_s_per_fold=("train_seconds","mean"),
    total_validation_inference_s=("validation_inference_seconds","sum"),
    trainable_parameters=("trainable_parameters","max"),
)
display(costs.round(3))

# 2. Plot wall-clock cost without implying that hardware differs by model.
fig,axes=plt.subplots(1,2,figsize=(15,5))
axes[0].barh(costs["model"],costs["total_train_s"]/60)
axes[0].set(xlabel="Total training minutes",title="Actual benchmark compute cost")
neural=costs[costs["model"]!="Random Forest"]
axes[1].barh(neural["model"],neural["trainable_parameters"])
axes[1].set(xlabel="Trainable parameters",title="Neural architecture capacity")
for ax in axes: ax.grid(axis="x",alpha=0.2)
fig.tight_layout()
plt.show()
print("Stored source fingerprint:",plan["fingerprint"])
print("Model scores reproduced from actual saved held-out predictions: YES")

**11. Research Interpretation and Decision Criteria for Stage 24**

This notebook supplies evidence for an architecture shortlist but not a final model choice or universal ranking. Strong development Macro F1 alone is insufficient: minority-class recall, consistency between folds and participants, training cost and stability across random seeds also matter. Stage 24 will retest a justified shortlist using multiple seeds and carefully isolated loss/imbalance interventions such as Focal Loss. **Any later selection must use Development only**, with S023/S032 held out from further model development.

In [ ]:
# ==================== 11. Evidence-Bounded Model Interpretation ====================

# 1. Report the full actual ranking with minority-class F1 and compute.
shuffling=class_results[class_results["class_code"]==3][["model","f1"]].rename(
    columns={"f1":"shuffling_pooled_f1"}
)
decision=summary.merge(shuffling,on="model",validate="one_to_one")
display(decision.sort_values("macro_f1_mean",ascending=False)[[
    "model","macro_f1_mean","macro_f1_sd",
    "balanced_accuracy_mean","shuffling_pooled_f1",
    "total_train_seconds","parameters"
]].round(4))

# 2. Specify limitations and the next evaluation objective explicitly.
limitations=pd.DataFrame([
    {"issue":"Development-only model comparison","restriction":"No independent Test claim"},
    {"issue":"500-cap label-blind sampling","restriction":"Not full-data optimum or class-balanced task"},
    {"issue":"One seed and three folds","restriction":"Cannot claim strong stability or significance"},
    {"issue":"Different architecture sizes and representations","restriction":"Not isolated causal architecture advantage"},
    {"issue":"Window overlap","restriction":"No independence of window-level observations"},
    {"issue":"Original Test exploratory history","restriction":"S023/S032 not preregistered-unseen"},
])
display(limitations)

**12. Final Source-Provenance and Scientific Quality Gate**

This gate rejects incomplete jobs, missing source-window predictions, duplicated held-out participants per model, non-finite metrics and any inclusion of protected Test people. A passing result means the seven *configured* models were actually evaluated on identical source rows using a subject-disjoint Development protocol. It does **not** prove external generalisation, superior performance on unseen settings, or final Test validation. Stage 24 will address multi-seed stability and class imbalance using the results generated here.

In [ ]:
# ==================== 12. Research Quality Gate ====================

# 1. Verify complete seven-model/three-fold runs from original source rows.
assert len(jobs)==len(MODEL_NAMES)*FOLDS==21
assert len(fold_results)==21
assert len(predictions)==len(X)*len(MODEL_NAMES)
assert len(subject_results)==len(DEVELOPMENT_SUBJECTS)*len(MODEL_NAMES)
assert len(class_results)==len(MODEL_NAMES)*len(CLASSES)

# 2. Assert every participant and every original window is evaluated per model.
for name,table in predictions.groupby("model"):
    assert len(table)==len(X) and table["source_row"].is_unique
    assert not(set(table["subject_id"])&RESERVED_TEST)
    np.testing.assert_array_equal(
        table.sort_values("source_row")["true_label"].to_numpy(),y
    )
assert fold_results["macro_f1"].between(0,1).all()
assert class_results["f1"].between(0,1).all()
assert all(confusion[name].shape==(8,8) for name in MODEL_NAMES)

# 3. Print reproducible scientific results and clear completion boundaries.
print("\n"+"="*70)
print("STAGE 23 | SEVEN-MODEL MATCHED GROUPED DEVELOPMENT BENCHMARK")
print("="*70)
print("Seven model families evaluated:",len(MODEL_NAMES))
print("Complete model/fold jobs:",len(jobs))
print("Independent Development participants:",len(DEVELOPMENT_SUBJECTS))
print("Validation source rows per model:",len(X))
print("Fixed eight-class metrics: PASSED")
print("Matched source-row and subject-fold identities: PASSED")
print("All seven actual training/evaluation runs: PASSED")
print("Original protected Test participants accessed: NO")
print("-"*70)
print("Stage 23 scientific quality gate: PASSED")
print("Independent full-dataset generalisation established: NO")
print("Final model selection performed: NO")
print("Next stage: Stage 24 - Multi-Seed and Class-Imbalance Robustness")
print("="*70)